In [115]:
import pandas as pd

data = pd.read_csv("../data/risks.csv")

In [116]:
#definitions
risk_priority = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

data["risk_priority"] = data["risk_level"].map(risk_priority)

strategies = {
    "Conservative": {
        "max_risk": 1,
        "min_act_timeline": 91,
        "sort_by": ["risk_priority", "op_reliab"],
        "ascending": [True, False]
    },
    "Balanced": {
        "max_risk": 3,
        "min_act_timeline": 41,
        "max_act_timeline": 90,
        "sort_by": ["startup_cost", "month_op_cost", "risk_priority"],
        "ascending": [True, True, True]
    },
    "Accelerated": {
        "max_risk": 2,
        "max_act_timeline": 40,
        "sort_by": ["rec_perf_ratio", "op_reliab"],
        "ascending": [False, False]
    }
}

In [117]:
#selection function
def select_sites(data, strategy_name, n_sites):
    strategy = strategies[strategy_name]
    eligible_sites = data.copy()
    eligible_sites = eligible_sites[
        eligible_sites["risk_priority"] <= strategy["max_risk"]
    ]

    if "min_act_timeline" in strategy:
        eligible_sites = eligible_sites[
            eligible_sites["act_timeline"] >= strategy["min_act_timeline"]
        ]

    if "max_act_timeline" in strategy:
        eligible_sites = eligible_sites[
            eligible_sites["act_timeline"] <= strategy["max_act_timeline"]
        ]

    selected_sites = (
        eligible_sites
        .sort_values(
            by=strategy["sort_by"],
            ascending=strategy["ascending"]
        )
        .head(n_sites)
        .copy()
    )

    return selected_sites

In [118]:
#selection strategy 
selected_by_strategy = {}

for strategy_name in strategies:
    selected_by_strategy[strategy_name] = select_sites(
        data,
        strategy_name,
        n_sites
    )

In [ ]:
strategy_results = []

for strategy_name, sites in selected_by_strategy.items():
    strategy_results.append({
        "Strategy": strategy_name,
        "Sites selected": len(sites),
        "Median activation timeline (days)": sites["act_timeline"].median(),
        "Median recruitment performance": sites["rec_perf_ratio"].median(),
        "Median operational reliability": sites["op_reliab"].median(),
        "Total startup cost": sites["startup_cost"].sum(),
        "Total monthly operating cost": sites["month_op_cost"].sum(),
        "High-risk sites": (sites["risk_level"] == "High").sum()
    })

results = pd.DataFrame(strategy_results)

results

,Strategy,Sites selected,Median activation timeline (days),Median recruitment performance,Median operational reliability,Total startup cost,Total monthly operating cost,High-risk sites
0,Conservative,30,95.0,1.02,0.90,218095,77663,0
1,Balanced,30,82.0,1.00,0.65,170935,73862,7
2,Accelerated,30,36.0,1.15,0.80,351747,112439,0
